In [58]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder,StandardScaler,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score,roc_auc_score
from collections import defaultdict
from sklearn.model_selection import GridSearchCV

In [31]:
df = pd.read_csv('data/WA_Fn-UseC_-Telco-Customer-Churn.xls')

In [32]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [33]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [34]:
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].mean())

In [35]:
# df['HasFamily'] = ((df['Partner'] == 'Yes') | (df['Dependents'] == 'Yes')).map({True: 'Yes', False: 'No'})

In [36]:
#df['Is_streaming'] = ((df['StreamingMovies'] == 'Yes') | (df['StreamingTV'] == 'Yes')).map({True:'Yes',False:'No'})

df['Is_streaming'] = np.where(
    (df['StreamingMovies'] == 'Nointernetservice') | (df['StreamingTV'] == 'Nointernetservice'), 'Nointernetservice',
        np.where(
            (df['StreamingMovies'] == 'Yes') | (df['StreamingTV'] == 'Yes'), 'Yes', 'No'))

In [37]:
# df['Monthly_charges_flag'] = np.where((df['MonthlyCharges']<=40),'LowMonthlyCharges',
#                                       np.where((df['MonthlyCharges']>40) & (df['MonthlyCharges'] <= 70),'MediumMonthlyCharges','HighMonthlyCharges'))

In [38]:
#df['Online_backup_security'] = ((df['OnlineBackup'] == 'Yes') | (df['OnlineSecurity'] == 'Yes')).map({True:"Yes",False:'No'})
# df['Online_backup_security'] = np.where(
#     (df['OnlineSecurity'] == 'Nointernetservice') | (df['OnlineBackup'] == 'Nointernetservice'), 'Nointernetservice',
#         np.where(
#             (df['OnlineSecurity'] == 'Yes') | (df['OnlineBackup'] == 'Yes'), 'Yes', 'No'))

In [39]:
# df['additional_services'] = np.where(((df['OnlineSecurity'] == 'Yes') | (df['OnlineBackup'] == 'Yes') | (df['DeviceProtection'] == 'Yes') | (df['TechSupport'] == 'Yes')),'Yes',
#                                      np.where(df['OnlineSecurity'] == 'Nointernetservice','Nointernetservice','No'))

In [40]:
drop_columns = ['customerID','StreamingMovies','StreamingTV']
for col in drop_columns:
    if col in df.columns:
        df.drop(columns=[col],inplace=True)
#df.drop(columns=['customerID','Partner','Dependents','StreamingMovies','StreamingTV'],inplace=True)
df.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'Contract',
       'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges',
       'Churn', 'Is_streaming'],
      dtype='str')

In [41]:
#df['PaymentMethod'] = (df['PaymentMethod'] == 'Electroniccheck').map({True:'Electroniccheck',False:'Other'})

In [42]:
X = df.drop(columns=['Churn'])
y = df['Churn']
X

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Is_streaming
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,No
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,Yes
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,One year,Yes,Credit card (automatic),103.20,7362.90,Yes
7040,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,No


In [43]:
numerical_feature = X.select_dtypes(exclude="str").columns
categorical_feature = X.select_dtypes(include="str").columns
categorical_feature

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'Is_streaming'],
      dtype='str')

In [44]:
numerical_feature

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='str')

In [45]:
for col in categorical_feature:
    df[col] =  df[col].str.replace(' ', '')
    df[col] =  df[col].str.replace('-', '')

In [46]:
X.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Is_streaming
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,No
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,No


In [47]:
preprocessor = ColumnTransformer([
    ('OHE',OneHotEncoder(drop='first'),categorical_feature),
    ('SC',StandardScaler(),numerical_feature)
])

In [48]:
le = LabelEncoder()

y = le.fit_transform(y)
y

array([0, 0, 1, ..., 0, 1, 0], shape=(7043,))

In [49]:
X = preprocessor.fit_transform(X)


In [50]:
X

array([[ 0.        ,  1.        ,  0.        , ..., -1.27744458,
        -1.16032292, -0.99497138],
       [ 1.        ,  0.        ,  0.        , ...,  0.06632742,
        -0.25962894, -0.17387565],
       [ 1.        ,  0.        ,  0.        , ..., -1.23672422,
        -0.36266036, -0.96039939],
       ...,
       [ 0.        ,  1.        ,  1.        , ..., -0.87024095,
        -1.1686319 , -0.85518222],
       [ 1.        ,  1.        ,  0.        , ..., -1.15528349,
         0.32033821, -0.87277729],
       [ 1.        ,  0.        ,  0.        , ...,  1.36937906,
         1.35896134,  2.01391739]], shape=(7043, 27))

In [51]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)
print(X_train)
print(y_train)

[[ 1.          1.          1.         ...  0.88073469  0.19736523
   0.65642602]
 [ 1.          0.          0.         ... -1.27744458  0.52473924
  -0.97258569]
 [ 1.          0.          0.         ... -0.78880022 -1.51096208
  -0.89350724]
 ...
 [ 1.          1.          1.         ... -0.82952058 -1.44947559
  -0.87302013]
 [ 1.          0.          0.         ... -0.82952058  1.15289851
  -0.47824601]
 [ 1.          0.          0.         ... -0.25943549 -1.49434411
  -0.80623836]]
[0 0 0 ... 0 1 0]


In [52]:
models = {
                "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
                "Random Forest": RandomForestClassifier(class_weight='balanced', n_estimators=200, random_state=42),
                "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
                "LightGBM": LGBMClassifier(class_weight='balanced', random_state=42),
                "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
                "Gradient Boosting": GradientBoostingClassifier(random_state=42),
                "SVM": SVC(class_weight='balanced', probability=True, random_state=42),
                "KNN": KNeighborsClassifier(n_neighbors=5)
            }

In [53]:
if 'report' not in locals():
    report = defaultdict(list)


In [54]:
for model_name,model in models.items():

    model.fit(X_train,y_train)

    predict = model.predict(X_test)

    roc_auc = roc_auc_score(y_test,predict)*100
    f1__score = f1_score(y_test,predict)*100

    report[model_name].append((roc_auc, f1__score))

    #report[model_name] = roc_auc,f1__score

[LightGBM] [Info] Number of positive: 1295, number of negative: 3635
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000306 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 633
[LightGBM] [Info] Number of data points in the train set: 4930, number of used features: 27
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


c:\Users\Dhvanish\OneDrive\Desktop\ML projects\Customer churn prediction\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [55]:
report

defaultdict(list,
            {'Logistic Regression': [(77.82713332563567, 64.87972508591065),
              (78.41758868716506, 65.36124240378123)],
             'Random Forest': [(74.13276868775371, 61.853978671041844),
              (73.61012060413003, 61.17065127782357)],
             'XGBoost': [(70.09563203401457, 56.695992179863154),
              (70.51045635769641, 57.33590733590733)],
             'LightGBM': [(76.60665892373208, 64.19570051890288),
              (75.37961887555383, 62.601028655400434)],
             'CatBoost': [(70.77930825256456, 57.84797630799605),
              (71.18235969327111, 58.48871442590775)],
             'Gradient Boosting': [(70.65971160964743, 57.65054294175715),
              (71.21343331227797, 58.58987090367428)],
             'SVM': [(77.43302474795843, 64.57461645746164),
              (78.15201961543424, 65.3287197231834)],
             'KNN': [(69.12380318456485, 55.06976744186046),
              (69.5843040301748, 55.74660633484163)]}

In [56]:

# for model, scores in sorted(report.items(), key=lambda item: item[1], reverse=True):
#     print(f"{model} -> roc_auc_score: {scores[0]:.4f}")
#     print(f"{model} -> f1_score: {scores[1]}")
#     print("\n")

table_data = []
for model_name, scores in report.items():
    # Fetch old and new scores safely
    old_roc, old_f1 = scores[0] if len(scores) >= 2 else (None, None)
    new_roc, new_f1 = scores[-1] if scores else (None, None)
    
    table_data.append({
        'Model Name': model_name,
        'Old ROC-AUC': old_roc,
        'New ROC-AUC': new_roc,
        'Old F1': old_f1,
        'New F1': new_f1
    })

# Convert to DataFrame and display
df_results = pd.DataFrame(table_data)
print(df_results)  # Or just type 'df_results' if in a Jupyter notebook cell


            Model Name  Old ROC-AUC  New ROC-AUC     Old F1     New F1
0  Logistic Regression    77.827133    78.417589  64.879725  65.361242
1        Random Forest    74.132769    73.610121  61.853979  61.170651
2              XGBoost    70.095632    70.510456  56.695992  57.335907
3             LightGBM    76.606659    75.379619  64.195701  62.601029
4             CatBoost    70.779308    71.182360  57.847976  58.488714
5    Gradient Boosting    70.659712    71.213433  57.650543  58.589871
6                  SVM    77.433025    78.152020  64.574616  65.328720
7                  KNN    69.123803    69.584304  55.069767  55.746606


In [57]:

# 1. Format the scores as strings "ROC / F1" for clean rows
formatted_report = {}
for model_name, scores in report.items():
    formatted_report[model_name] = [f"{roc:.4f} / {f1:.4f}" for roc, f1 in scores]

# 2. Create the DataFrame (this automatically sets model names as column headers)
# We use pd.DataFrame.from_dict with orient='columns' (default)
df_transposed = pd.DataFrame.from_dict(formatted_report, orient='columns')

# 3. Label the row index to represent the experiment runs
df_transposed.index = [f"Run {i+1}" for i in range(len(df_transposed))]

# Display the DataFrame
df_transposed


,Logistic Regression,Random Forest,XGBoost,LightGBM,CatBoost,Gradient Boosting,SVM,KNN
Run 1,77.8271 / 64.8797,74.1328 / 61.8540,70.0956 / 56.6960,76.6067 / 64.1957,70.7793 / 57.8480,70.6597 / 57.6505,77.4330 / 64.5746,69.1238 / 55.0698
Run 2,78.4176 / 65.3612,73.6101 / 61.1707,70.5105 / 57.3359,75.3796 / 62.6010,71.1824 / 58.4887,71.2134 / 58.5899,78.1520 / 65.3287,69.5843 / 55.7466


In [62]:
param_grid = {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l1','l2'], 'solver': ['liblinear']}
grid = GridSearchCV(LogisticRegression(class_weight='balanced'),scoring='f1',cv=10,param_grid=param_grid)

grid.fit(X_train,y_train)

grid_predict = grid.predict(X_test)

grid_f1 = f1_score(y_test,grid_predict)

print(grid.best_score_)
print(grid.best_params_)

c:\Users\Dhvanish\OneDrive\Desktop\ML projects\Customer churn prediction\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Dhvanish\OneDrive\Desktop\ML projects\Customer churn prediction\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\Dhvanish\OneDrive\Desktop\ML projects\Customer churn prediction\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed 

0.6127722709088981
{'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}


c:\Users\Dhvanish\OneDrive\Desktop\ML projects\Customer churn prediction\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Dhvanish\OneDrive\Desktop\ML projects\Customer churn prediction\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.